In [ ]:
!pip install datasets tiktoken --upgrade -qqq

In [ ]:
import torch
import torch.nn as nn
import transformers
import datasets
from datasets import load_dataset
from torch.utils.data import Dataset

import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
  ModelConfig = {
 "vocab_size": 50257,  # Kelime dağarcığı boyutu
 "context_length": 256,  # Bağlam uzunluğu
 "emb_dim": 384,  # Gömülü (embedding) boyutu
 "n_heads": 12,  # Dikkat başlıklarının (attention heads) sayısı
 "n_layers": 12,  # Katman sayısı
 "drop_rate": 0.1,  # Dropout oranı
 "qkv_bias": False  # Sorgu-Anahtar-Değer (Query-Key-Value) bias'ı
}

In [ ]:
class LayerNorm(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5  
        self.scale = nn.Parameter(torch.ones(emb_dim))  
        self.shift = nn.Parameter(torch.zeros(emb_dim)) 

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)

        var = x.var(dim=-1, keepdim=True, unbiased=False)

        norm_x = (x - mean) / torch.sqrt(var + self.eps)

        return self.scale * norm_x + self.shift

In [ ]:
class GELU(nn.Module):
     def __init__(self):
         super().__init__()
     def forward(self, x):
         return 0.5 * x * (1 + torch.tanh(
         torch.sqrt(torch.tensor(2.0 / torch.pi)) *
         (x + 0.044715 * torch.pow(x, 3))
         ))

class MLP(nn.Module):

    def __init__(self, config:ModelConfig):
        super().__init__()
        self.emb_dim = config["emb_dim"]
        self.vocab_size = config["vocab_size"]
        self.context_len = config["context_length"]

        self.layers = nn.Sequential(
            nn.Linear(self.emb_dim, 4 * self.emb_dim),
            GELU(),
            nn.Linear(self.emb_dim * 4, self.emb_dim)
        )

    def forward(self,x):
        return self.layers(x)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.emb_dim = config["emb_dim"]  # Gömme (embedding) boyutu
        self.qkv_bias = config["qkv_bias"]  # Sorgu, anahtar ve değer için bias kullanılıp kullanılmayacağı
        self.n_heads = config["n_heads"]  # Başlık sayısı (multi-head attention)
        self.context_length = config["context_length"]  # Bağlam uzunluğu (örneğin, dil modeli için cümledeki kelime sayısı)

        self.head_dim = self.emb_dim // self.n_heads
        assert (self.emb_dim % self.n_heads == 0), \
             "emb_dim must be divisible by num_heads"

        self.Wq = nn.Linear(self.emb_dim, self.emb_dim, bias=self.qkv_bias)
        self.Wk = nn.Linear(self.emb_dim, self.emb_dim, bias=self.qkv_bias)
        self.Wv = nn.Linear(self.emb_dim, self.emb_dim, bias=self.qkv_bias)
        self.out_proj = nn.Linear(self.emb_dim, self.emb_dim)

        self.register_buffer(
         "mask",
        torch.triu(torch.ones(self.context_length, self.context_length),
         diagonal=1
                    ))

    def forward(self, x):
        batch_size, seq_len, emb_dim = x.shape

        q = self.Wq(x)  # Sorgular (Query)
        k = self.Wk(x)  # Anahtarlar (Key)
        v = self.Wv(x)  # Değerler (Value)

        k = k.view(batch_size, seq_len, self.n_heads, self.head_dim)
        v = v.view(batch_size, seq_len, self.n_heads, self.head_dim)
        q = q.view(batch_size, seq_len, self.n_heads, self.head_dim)

        k = k.transpose(1, 2)
        q = q.transpose(1, 2)
        v = v.transpose(1, 2)

        attn_scores = q @ k.transpose(2, 3)  # (b, n_heads, seq_len, seq_len)

        mask_bool = self.mask.bool()[:seq_len, :seq_len]  # Mask'ın uygun kısmını al
        attn_scores.masked_fill_(mask_bool, -torch.inf)  # Geleceği görmesini engeller (sonsuz küçük bir değere ayarlama)

        attn_weights = torch.softmax(
                attn_scores / k.shape[-1]**0.5, dim=-1
                                        )  # Skorları normalize et, d_k (head_dim) ile ölçeklendir

        context_vec = (attn_weights @ v).transpose(1, 2)  # (b, n_heads, seq_len, head_dim) -> (b, seq_len, n_heads, head_dim)

        context_vec = context_vec.contiguous().view(
             batch_size, seq_len, self.emb_dim
                                                    )

        context_vec = self.out_proj(context_vec)  # Sonuçları projekte et

        return context_vec

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.att = MultiHeadAttention(config)
        self.ff = MLP(config)
        self.norm1 = LayerNorm(config["emb_dim"])  # İlk layer norm (attention öncesi)
        self.norm2 = LayerNorm(config["emb_dim"])  # İkinci layer norm (feedforward öncesi)

    def forward(self, x):
        shortcut = x

        x = self.norm1(x)  # İlk normalizasyon
        x = self.att(x)  # Multi-head attention uygulaması

        x = x + shortcut  # Kısa yol bağlantısı ekleniyor

        shortcut = x

        x = self.norm2(x)  # İkinci normalizasyon
        x = self.ff(x)  # Feedforward (MLP) işlemi

        x = x + shortcut  # Kısa yol bağlantısı ekleniyor

        return x

In [ ]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [ ]:
class GPTModel(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        # Token embedding tablosu: her token ID'si için d_model boyutunda vektörler.
        # Vocab boyutu (cfg["vocab_size"]) ve her token'in embedding boyutu (cfg["emb_dim"]) parametreleriyle oluşturuluyor.
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])

        # Positional embedding tablosu: her token'in cümledeki pozisyonunu temsil eden embedding.
        # Cümledeki en uzun token sayısı (cfg["context_length"]) ve embedding boyutu (cfg["emb_dim"]) parametreleriyle oluşturuluyor.
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

        # Transformer bloklarının ardışık dizisi (Sequential):
        # Burada modelin içinde cfg["n_layers"] kadar TransformerBlock bulunuyor.
        # Her TransformerBlock, attention ve feedforward katmanlarını içeriyor.
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        # Son layer normalization katmanı: Modelin çıktısı üzerinde normalizasyon uygulanarak, öğrenmenin stabilleşmesi sağlanır.
        self.final_norm = LayerNorm(cfg["emb_dim"])

        # Çıktı başlığı (output head): Sonuçları vocab boyutuna (cfg["vocab_size"]) eşleştiren bir lineer katman.
        # Bu katman, modelin final tahminlerini yapabilmesi için gereklidir.
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        # Girişin boyutlarını alıyoruz: (batch_size, seq_len)
        batch_size, seq_len = in_idx.shape

        # Token embeddingler: Her token ID'si için embeddingler alınıyor.
        tok_embeds = self.tok_emb(in_idx)

        # Positional embeddingler: Cümledeki her token'in pozisyonuna karşılık gelen embeddingler alınıyor.
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        # Token ve positional embeddingleri topluyoruz. Bu şekilde hem token'in anlamı hem de pozisyonu dikkate alınır.
        x = tok_embeds + pos_embeds
        # Transformer bloklarında geçiyor: Modelin temel özelliklerini öğrenebilmesi için.
        x = self.trf_blocks(x)

        # Son layer normalization: Çıktıları normalize ediyoruz.
        x = self.final_norm(x)

        # Çıktıyı lineer katmandan geçiriyoruz. Bu, modelin vocab_size kadar olasılık dağılımı oluşturmasını sağlar.
        logits = self.out_head(x)

        # Sonuç olarak, logits (tahminler) döndürülüyor.
        return logits


In [ ]:
model = GPTModel(ModelConfig)
def generate_text_simple(model, idx,
            max_new_tokens, context_size):
    for i in range(max_new_tokens):
        idx_cound = idx[:,-context_size:]
        with torch.no_grad():
            logits = model(idx_cound)

        logits = logits[:,-1,:]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx,idx_next), dim=1)

    return idx

In [ ]:
checkpoint = torch.load('/kaggle/input/turna54k/pytorch/default/1/model_checkpoint_adim_54000.pth')

model = GPTModel(ModelConfig)

model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
start_context = "Merhaba ben"
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
model.eval()
out = generate_text_simple(
 model=model,
 idx=encoded_tensor,
 max_new_tokens=60,
 context_size=ModelConfig["context_length"]
)

decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

In [ ]:
# Model parametrelerini geri yükleyin
dataset = load_dataset('merve/turkish_instructions')

In [ ]:
dataset

In [ ]:
print(dataset['train'].column_names)

In [ ]:
print(dataset['train'][' çıktı'][0])

In [ ]:
train_dataset = dataset['train']
len(train_dataset)

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Aşağıda bir görevi tanımlayan bir talimat bulunmaktadır. "
        f"Lütfen bu talimata uygun bir yanıt yazınız."
        f"\n\n### Talimat:\n{entry['talimat']}"
    )
    input_text = (
        f"\n\n### Girdi:\n{entry[' giriş']}" if entry[" giriş"] else ""
    )
    return instruction_text + input_text

In [ ]:
model_input = format_input(train_dataset)
desired_response = f"\n\n### Cevap:\n{train_dataset[' çıktı']}"

In [ ]:
model_input = format_input(train_dataset[999])
desired_response = f"\n\n### Cevap:\n{train_dataset[999][' çıktı']}"
print(model_input + desired_response)

In [ ]:
train_portion = int(len(train_dataset) * 0.85)
test_portion = int(len(train_dataset) * 0.1)
val_portion = len(train_dataset) - train_portion - test_portion

print(train_portion, test_portion, val_portion)

In [ ]:
train_portion = int(len(train_dataset) * 0.85)

train_data = train_dataset.select(range(train_portion))
len(train_data)

In [ ]:
print(type(train_dataset))


In [ ]:
from torch.utils.data import random_split

train_size = int(0.90 * len(train_dataset))
val_size = len(train_dataset) - train_size 

train_data, val_data = random_split(
    train_dataset,
    [train_size, val_size]
)

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))


In [ ]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry[' çıktı']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

In [ ]:
def custom_collate(
    batch,
    pad_token_id=50256,
    device="cuda",
    ignore_index= -100,
    allowed_max_len = None
):
    batch_max_len = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (
            new_item + [pad_token_id] * (batch_max_len - len(new_item))
        )
        
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        max_length = ModelConfig["context_length"]
        inputs = torch.tensor(padded[:-1])[:max_length]  
        targets = torch.tensor(padded[1:])[:max_length]
        
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_len is not None:
            inputs = inputs[:allowed_max_len]
            targets = targets[:allowed_max_len]
        
        targets_lst.append(targets)
        inputs_lst.append(inputs)
    
    input_tensor = torch.stack(inputs_lst).to(device)
    target_tensor = torch.stack(targets_lst).to(device)
    
    return input_tensor, target_tensor

**Replace certain padding tokens by -100 to exclude them from the training loss.**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
from functools import partial
customized_collate_fn = partial(
 custom_collate,
 device=device,
 allowed_max_len=1024
)

In [ ]:
from torch.utils.data import DataLoader
num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
     train_dataset,
     batch_size=batch_size,
     collate_fn=customized_collate_fn,
     shuffle=True,
     drop_last=True,
     num_workers=num_workers
)
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
     val_dataset,
     batch_size=batch_size,
     collate_fn=customized_collate_fn,
     shuffle=False,
     drop_last=False,
     num_workers=num_workers
)

In [ ]:
torch.manual_seed(123)
input_text = format_input(val_data[0])
print(input_text)

In [ ]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()

In [ ]:
def generate(model, idx, max_new_tokens, context_size,
            temprature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:,-context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:,-1,:]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:,-1]
            logits = torch.where(
                logits < min_val,
                torch.tensor(float('-inf')).to(logits.device),
                logits
            )

        if temprature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if idx_next == eos_id:
            break
        idx = torch.cat((idx, idx_next), dim=-1)

    return idx

In [ ]:
token_ids = generate(
 model=model,
 idx=text_to_token_ids(input_text, tokenizer),
 max_new_tokens=35,
 context_size=ModelConfig["context_length"],
 eos_id=50256,
)

token_ids = token_ids.to(device)
generated_text = token_ids_to_text(token_ids, tokenizer)

In [ ]:
response_text = generated_text[len(input_text):].strip()
print(response_text)

In [ ]:
print(generated_text)

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
    logits.flatten(0, 1), target_batch.flatten()
    )
    return loss

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(
            input_batch, target_batch, model, device
            )
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
def train_model_simple(model, train_loader, val_loader,
                        optimizer, device, num_epochs,
                        eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % 10000 == 0 and global_step != 0:
                checkpoint = {
                    'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'global_step': global_step,
                    'tokens_seen': tokens_seen,
                    'train_losses': train_losses,
                    'val_losses': val_losses
                }
                torch.save(checkpoint, f'model_checkpoint_step_{global_step}.pth')
                print(f"{global_step} adımda model kaydedildi: model_checkpoint_step_{global_step}.pth")

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)

                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {train_loss:.3f}, "
                    f"Val loss {val_loss:.3f}"
                )
                allocated = torch.cuda.memory_allocated(device) / 1024**2
                reserved = torch.cuda.memory_reserved(device) / 1024**2
                


        generate_and_print_sample(
            model, tokenizer, device, start_context
        )
        # Epoch sonunda modeli kaydetme
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'global_step': global_step,
            'tokens_seen': tokens_seen,
            'train_losses': train_losses,
            'val_losses': val_losses
        }
        torch.save(checkpoint, f'model_checkpoint_epoch_{epoch+1}.pth')
        print(f"Model kaydedildi: model_checkpoint_epoch_{epoch+1}.pth")
    return train_losses, val_losses, track_tokens_seen

In [ ]:
model.to(device)
torch.manual_seed(123)

In [ ]:
with torch.no_grad():
 train_loss = calc_loss_loader(
     train_loader, model, device, num_batches=5
 )
 val_loss = calc_loss_loader(
     val_loader, model, device, num_batches=5
)
    
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
import time
start_time = time.time()
torch.manual_seed(123)

optimizer = torch.optim.AdamW(
     model.parameters(), lr=0.00005, weight_decay=0.1
)
num_epochs = 5

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(  # Burada düzeltme yapıldı
            val_loader, model, device, num_batches=eval_iter
        )
    model.train()
    return train_loss, val_loss

In [ ]:
train_losses, val_losses, tokens_seen = train_model_simple(
 model, train_loader, val_loader, optimizer, device,
 num_epochs=num_epochs, eval_freq=50, eval_iter=50,
 start_context=format_input(val_data[0]), tokenizer=tokenizer
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
 fig, ax1 = plt.subplots(figsize=(5, 3))
 ax1.plot(epochs_seen, train_losses, label="Training loss")
 ax1.plot(
 epochs_seen, val_losses, linestyle="-.", label="Validation loss"
 )
 ax1.set_xlabel("Epochs")
 ax1.set_ylabel("Loss")
 ax1.legend(loc="upper right")
 ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
 ax2 = ax1.twiny()
 ax2.plot(tokens_seen, train_losses, alpha=0)
 ax2.set_xlabel("Tokens seen")
 fig.tight_layout()
 plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)